# 4 inference data creation

Create inference dataset: read test files, apply the same preprocessing & processing pipeline, sample 10k rows, save as Parquet, then load the Parquet and write JSON (records, newline-delimited).

In [ ]:
import pandas as pd
from pathlib import Path
import gc
import os

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT == REPO_ROOT.parent:
        raise FileNotFoundError("Could not find pyproject.toml in the current directory or its parents.")
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
print(REPO_ROOT)


In [ ]:
from src.data_processing.preprocessing import (
    run_preprocessing,
    run_processing_pipeline,
    split_train_validation_test,
)
from src.run_time_configuration import BaseConfigParams


In [ ]:
# Fit the processing pipeline on the original training window (same as 01_data_transformation)
IEEE_DIR = Path("data/raw/ieee-fraud-detection")

# Load train data to fit the pipeline
train_tx = pd.read_csv(IEEE_DIR / "train_transaction.csv")
id_tx = pd.read_csv(IEEE_DIR / "train_identity.csv").assign(_has_identity=True)
combined_train = train_tx.join(
    id_tx.set_index("TransactionID"),
    on="TransactionID",
    how="left",
)
del train_tx, id_tx
gc.collect()

combined_train = run_preprocessing(combined_train)

runtime_config = BaseConfigParams(
    experiment_name="ieee_fraud",
    mlflow_run_name="ieee_inference_pipeline_fit",
    training_start_date="2017-12-01",
    training_end_date="2018-03-31",
    test_start_date="2018-04-01",
    test_end_date="2018-06-30",
)

df_train, df_validation, _ = split_train_validation_test(
    combined_train,
    runtime_config,
)
pipeline, df_train_proc, df_validation_proc, _ = run_processing_pipeline(
    df_train,
    df_validation,
)
print("Pipeline fitted. Train shape:", df_train_proc.shape)


In [ ]:
# Load test data, apply preprocessing and the fitted pipeline
test_tx = pd.read_csv(IEEE_DIR / "test_transaction.csv")
test_id = pd.read_csv(IEEE_DIR / "test_identity.csv").assign(_has_identity=True)
combined_test = test_tx.join(
    test_id.set_index("TransactionID"),
    on="TransactionID",
    how="left",
)
del test_tx, test_id
gc.collect()

combined_test = run_preprocessing(combined_test)
df_test_proc = pipeline.transform(combined_test)
print("Processed test shape:", df_test_proc.shape)


In [ ]:
# Sample up to 10,000 rows for the inference dataset
SAMPLE_N = 10000
if len(df_test_proc) > SAMPLE_N:
    df_inference = df_test_proc.sample(SAMPLE_N, random_state=42)
else:
    df_inference = df_test_proc.copy()

print("Inference sample shape:", df_inference.shape)

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
parquet_path = PROCESSED_DIR / "inference.parquet"
df_inference.to_parquet(parquet_path, index=False)
print("Wrote inference parquet:", parquet_path.resolve())


In [ ]:
# Load the Parquet file and write newline-delimited JSON records
df_loaded = pd.read_parquet(parquet_path)
json_path = PROCESSED_DIR / "inference.json"
df_loaded.to_json(json_path, orient="records", lines=True)
print("Wrote inference JSON:", json_path.resolve())


In [ ]:
df_loaded